In [5]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

with open("../../sql/queries/model_training_features.sql", "r") as f:
    query = f.read()

df = pd.read_sql(query, engine)
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)
df.head()

(80237, 14)


,transaction_id,customer_id,timestamp,amount,country,channel,fraud_label,fraud_type,transactions_last_60_sec,country_changed,amount_last_7_days,merchant_risk_score,merchant_category,customer_risk_level
0,defe8df2-4cd7-4ee6-822d-eda2468389b1,CUST_0003842,2024-01-01 00:01:01,28.77,BR,mobile,0,none,1,None,28.77,0.000,utilities,medium
1,24fc0440-e8a5-4370-9f04-e937ad35c884,CUST_0004463,2024-01-01 00:04:14,36.84,AE,online,0,none,1,None,36.84,0.320,travel,medium
2,f26ad290-df78-4aa5-83a6-332efa9f4618,CUST_0003923,2024-01-01 00:08:13,248.89,BR,pos,0,none,1,None,248.89,0.586,crypto,medium
3,dfb1637f-68de-4ff4-b7d5-793bd0ec481c,CUST_0000753,2024-01-01 00:13:33,94.27,NG,pos,0,none,1,None,94.27,0.128,electronics,low
4,638c60c5-f3c9-44b8-8187-74bf9f9fafff,CUST_0000892,2024-01-01 00:18:59,0.09,CN,mobile,0,none,1,None,0.09,0.146,grocery,medium


In [6]:
df["hour_of_day"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek

country_changed_dummies = pd.get_dummies(df["country_changed"], prefix="country_changed", dummy_na=True)
country_dummies = pd.get_dummies(df["country"], prefix="country")
channel_dummies = pd.get_dummies(df["channel"], prefix="channel")
merchant_category_dummies = pd.get_dummies(df["merchant_category"], prefix="merchant_cat")
customer_risk_dummies = pd.get_dummies(df["customer_risk_level"], prefix="customer_risk")

df = pd.concat([
    df, country_changed_dummies, country_dummies, channel_dummies,
    merchant_category_dummies, customer_risk_dummies
], axis=1)

feature_columns = (
    ["amount", "hour_of_day", "day_of_week", "transactions_last_60_sec",
     "amount_last_7_days", "merchant_risk_score"]
    + list(country_changed_dummies.columns)
    + list(country_dummies.columns)
    + list(channel_dummies.columns)
    + list(merchant_category_dummies.columns)
    + list(customer_risk_dummies.columns)
)

X = df[feature_columns]
y = df["fraud_label"]

print(X.shape)

(80237, 33)


In [3]:
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"--- Fold {fold} (baseline) ---")
    print(classification_report(y_test, preds, digits=3))

--- Fold 0 (baseline) ---
              precision    recall  f1-score   support

           0      0.998     1.000     0.999     13318
           1      0.842     0.593     0.696        54

    accuracy                          0.998     13372
   macro avg      0.920     0.796     0.847     13372
weighted avg      0.998     0.998     0.998     13372

--- Fold 1 (baseline) ---
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     13349
           1      0.815     0.957     0.880        23

    accuracy                          1.000     13372
   macro avg      0.907     0.978     0.940     13372
weighted avg      1.000     1.000     1.000     13372

--- Fold 2 (baseline) ---
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     13342
           1      0.818     0.900     0.857        30

    accuracy                          0.999     13372
   macro avg      0.909     0.950     0.928     13

In [7]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model = XGBClassifier(eval_metric="logloss", scale_pos_weight=scale_pos_weight)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"--- Fold {fold} (weighted, scale_pos_weight={scale_pos_weight:.1f}) ---")
    print(classification_report(y_test, preds, digits=3))

--- Fold 0 (weighted, scale_pos_weight=192.9) ---
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     13318
           1      0.906     0.889     0.897        54

    accuracy                          0.999     13372
   macro avg      0.953     0.944     0.948     13372
weighted avg      0.999     0.999     0.999     13372

--- Fold 1 (weighted, scale_pos_weight=216.5) ---
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     13349
           1      0.821     1.000     0.902        23

    accuracy                          1.000     13372
   macro avg      0.911     1.000     0.951     13372
weighted avg      1.000     1.000     1.000     13372

--- Fold 2 (weighted, scale_pos_weight=273.8) ---
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     13342
           1      0.933     0.933     0.933        30

    accuracy                    